# Kannada Character Recognition - Run Project

This notebook provides a step-by-step guide to running the Kannada Character Recognition model. It handles all necessary setup, including creating the class mapping and loading the trained model.

## Prerequisites

Ensure you have the following files/folders in your project directory (`C:\Users\user\Documents\Kannadatest`):

1.  **`best_model.pth`**: The trained model file. (If missing, you must run the training notebook first).
2.  **`chars74k/`**: The dataset directory containing images.
3.  **`Maps/Kannada/Img/source_image_map.txt`**: The mapping file from the dataset.

## Step 1: Setup and Imports

Run the following cell to import necessary libraries and set up the device.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import json
import os
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

: 

## Step 2: Generate Class Mapping

We need a file `class_mapping.json` that tells us which class ID corresponds to which Kannada character. This cell checks if it exists, and if not, creates it from `source_image_map.txt`.

In [ ]:
mapping_file = "class_mapping.json"
source_map_file = r"Maps\Kannada\Img\source_image_map.txt"

def create_mapping(source_path, output_path):
    print(f"Generating mapping from {source_path}...")
    mapping = {}
    try:
        with open(source_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split(';')
                if len(parts) >= 3:
                    image_path_part = parts[0]
                    character = parts[2]
                    if "Sample" in image_path_part:
                        sample_part = image_path_part.split('/')[2]
                        if sample_part.startswith("Sample"):
                            try:
                                sample_id = int(sample_part.replace("Sample", ""))
                                class_index = sample_id - 1
                                mapping[class_index] = character
                            except ValueError:
                                continue
        
        sorted_mapping = dict(sorted(mapping.items()))
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(sorted_mapping, f, ensure_ascii=False, indent=4)
        print(f"✅ Successfully created {output_path} with {len(mapping)} classes.")
    except FileNotFoundError:
        print(f"❌ Error: Source mapping file not found at {source_path}")
        print("Please ensure you have extracted the dataset correctly.")

if not os.path.exists(mapping_file):
    create_mapping(source_map_file, mapping_file)
else:
    print("✅ class_mapping.json already exists.")

# Load the mapping
id2char = {}
if os.path.exists(mapping_file):
    with open(mapping_file, "r", encoding="utf-8") as f:
        loaded_mapping = json.load(f)
        id2char = {int(k): v for k, v in loaded_mapping.items()}
    print(f"Loaded mapping for {len(id2char)} classes.")
else:
    print("❌ Failed to load mapping. Cannot proceed with correct labels.")

## Step 3: Load the Model

Now we load the trained ResNet18 model from `best_model.pth`.

In [ ]:
model_path = "best_model.pth"
num_classes = 657

model = models.resnet18(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    print("✅ Model loaded successfully.")
else:
    print(f"❌ Error: {model_path} not found. Please run the training notebook to generate the model file.")

## Step 4: Predict on an Image

Run this cell to predict the character for a specific image. You can change `image_path` to test other images.

In [ ]:
# Change this path to test different images
image_path = r"chars74k\Kannada\Hnd\Img\Sample001\img001-001.png"

def predict_image(img_path):
    if not os.path.exists(img_path):
        print(f"❌ Image not found: {img_path}")
        return

    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])

    try:
        image = Image.open(img_path).convert("RGB")
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(input_tensor)
            _, predicted = outputs.max(1)
            predicted_class = predicted.item()

        predicted_char = id2char.get(predicted_class, f"Class_{predicted_class+1}")
        
        plt.imshow(image)
        plt.axis("off")
        plt.title(f"Predicted: {predicted_char} (Class {predicted_class})")
        plt.show()
        
        print(f"Predicted Character: {predicted_char}")
        
    except Exception as e:
        print(f"Error: {e}")

predict_image(image_path)